Copyright (c) MONAI Consortium  
Licensed under the Apache License, Version 2.0 (the "License");  
you may not use this file except in compliance with the License.  
You may obtain a copy of the License at  
&nbsp;&nbsp;&nbsp;&nbsp;http://www.apache.org/licenses/LICENSE-2.0  
Unless required by applicable law or agreed to in writing, software  
distributed under the License is distributed on an "AS IS" BASIS,  
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.  
See the License for the specific language governing permissions and  
limitations under the License.

You can download and run this notebook locally, or you can run it for free in a cloud environment using Colab or Sagemaker Studio Lab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Project-MONAI/tutorials/blob/main/modules/idc_dataset.ipynb)

[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github.com/Project-MONAI/tutorials/blob/main/modules/idc_dataset.ipynb)

# Quick Start With IDC Data Using `idc-index`

[NCI Imaging Data Commons (IDC)](https://imaging.datacommons.cancer.gov/) hosts all public [TCIA](https://www.cancerimagingarchive.net/) DICOM datasets in cloud storage (AWS S3 / Google Cloud Storage), providing free, no-authentication-required programmatic access to thousands of cancer imaging collections.

This tutorial mirrors the workflow in `tcia_dataset.ipynb` — using the same **QIN-PROSTATE-Repeatability** prostate MRI collection with DICOM segmentations — but replaces the TCIA REST API and `TciaDataset` with the [`idc-index`](https://github.com/ImagingDataCommons/idc-index) Python package and a custom `IDCDataset` class built on MONAI's `CSVDataset`.

We'll cover the following topics:
* Discover collections with DICOM segmentation data in IDC using SQL queries
* Explore collection metadata with `idc-index`
* Build an `IDCDataset` that pairs MR image series with their DICOM SEG series
* Visualize images and segmentation masks
* Create a full segmentation training pipeline with MONAI + PyTorch

## Setup environment

In [ ]:
!python -c "import monai" || pip install -q "monai-weekly[nibabel, pillow, ignite, tqdm, pydicom]"
!python -c "import matplotlib" || pip install -q matplotlib
!pip install -q --upgrade idc-index
%matplotlib inline

## Setup imports

In [ ]:
import os
import random
import shutil
import tempfile
from typing import Callable, Optional

import matplotlib.pyplot as plt
import pandas as pd
import torch

import monai
from idc_index import IDCClient
from monai.config import print_config
from monai.data import CSVDataset, DataLoader, decollate_batch
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import (
    Activations,
    AsDiscrete,
    Compose,
    EnsureChannelFirstd,
    LoadImaged,
    NormalizeIntensityd,
    Orientationd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    ResampleToMatchd,
    Spacingd,
)

print_config()

## Setup data directory

You can specify a directory with the `MONAI_DATA_DIRECTORY` environment variable.  
This allows you to save results and reuse downloads.  
If not specified a temporary directory will be used.

In [ ]:
directory = os.environ.get("MONAI_DATA_DIRECTORY")
if directory is not None:
    os.makedirs(directory, exist_ok=True)
root_dir = tempfile.mkdtemp() if directory is None else directory
print(root_dir)

## Brief Introduction

IDC is a cloud-based repository that mirrors all public TCIA DICOM datasets in AWS S3 and Google Cloud Storage. Data can be queried and downloaded without any authentication using the `idc-index` Python package.

IDC organizes data into *collections* (equivalent to TCIA collections). The primary metadata index is queried with standard SQL. A specialized `seg_index` table tracks DICOM Segmentation (SEG) objects and links them back to their source image series via the `segmented_SeriesInstanceUID` column — making it straightforward to discover which collections have segmentation data and to pair images with their segmentations.

You can explore available collections interactively at the [IDC Portal](https://portal.imaging.datacommons.cancer.gov/explore/) or programmatically as shown below.

In [ ]:
# Initialize the IDC client and load the seg_index table
client = IDCClient()
client.fetch_index("seg_index")

# Find all collections that have DICOM SEG objects paired with MR images
seg_collections = client.sql_query("""
    SELECT
        src.collection_id,
        COUNT(DISTINCT src.PatientID) AS patients_with_seg
    FROM seg_index seg
    JOIN index src ON seg.segmented_SeriesInstanceUID = src.SeriesInstanceUID
    WHERE src.Modality = 'MR'
    GROUP BY src.collection_id
    ORDER BY patients_with_seg DESC
""")
print(seg_collections.to_string(index=False))

## Collection Details Lookup

Once you identify a collection of interest, you can query its metadata directly from the IDC index. Here we inspect the **QIN-PROSTATE-Repeatability** collection (IDC collection ID: `qin_prostate_repeatability`), the same dataset used in `tcia_dataset.ipynb`.

In [ ]:
collection_id = "qin_prostate_repeatability"

info = client.sql_query(f"""
    SELECT
        COUNT(DISTINCT PatientID)      AS patients,
        GROUP_CONCAT(DISTINCT Modality, ', ') AS modalities,
        GROUP_CONCAT(DISTINCT BodyPartExamined, ', ') AS body_parts
    FROM index
    WHERE collection_id = '{collection_id}'
""")
print(info.to_string(index=False))

## Define IDCDataset

`IDCDataset` inherits from MONAI's `CSVDataset` and handles three responsibilities:

1. **Query IDC metadata** — joins `seg_index` with `index` to build a CSV of paired `(image SeriesInstanceUID, SEG SeriesInstanceUID)` rows for the requested collection.
2. **Cache the CSV** — on subsequent runs the CSV is reused without re-querying IDC.
3. **Download DICOM series on demand** — `__getitem__` calls `IDCClient.download_from_selection()` the first time a series is accessed; subsequent accesses reuse the local files.

Key parameters:
- **collection_id**: IDC collection identifier (e.g. `"qin_prostate_repeatability"`)
- **img_dir**: local directory to store downloaded DICOM series
- **section**: `"training"` or `"validation"`
- **val_frac**: fraction of cases reserved for validation (default 0.2)
- **transform**: MONAI transform pipeline applied to each loaded item

In [ ]:
class IDCDataset(CSVDataset):
    """
    Dataset that pairs MR image series with their DICOM SEG series from IDC.

    Queries seg_index + index to build a CSV with one row per
    (image_SeriesInstanceUID, seg_SeriesInstanceUID) pair, then downloads
    DICOM files on demand via idc-index.

    Args:
        collection_id: IDC collection identifier (e.g. "qin_prostate_repeatability").
        img_dir: root directory for downloaded DICOM series.
        section: ``"training"`` or ``"validation"``.
        val_frac: fraction of data reserved for validation.
        seed: random seed for train/val split.
        csv_path: path to cache the paired metadata CSV. Re-used on subsequent runs.
        transform: transform applied to each loaded item.
    """

    def __init__(
        self,
        collection_id: str,
        img_dir: str,
        section: str = "training",
        val_frac: float = 0.2,
        seed: int = 0,
        csv_path: Optional[str] = None,
        transform: Optional[Callable] = None,
    ):
        self.client = IDCClient()
        self.client.fetch_index("seg_index")
        self.img_dir = img_dir
        os.makedirs(img_dir, exist_ok=True)

        if csv_path is None:
            csv_path = os.path.join(img_dir, f"{collection_id}_paired.csv")

        if not os.path.exists(csv_path):
            df = self.client.sql_query(f"""
                SELECT
                    src.PatientID,
                    src.StudyInstanceUID,
                    src.SeriesInstanceUID  AS image_SeriesInstanceUID,
                    seg.SeriesInstanceUID  AS seg_SeriesInstanceUID
                FROM seg_index seg
                JOIN index src
                  ON seg.segmented_SeriesInstanceUID = src.SeriesInstanceUID
                WHERE src.collection_id = '{collection_id}'
                  AND src.Modality = 'MR'
                ORDER BY src.PatientID
            """)
            df.to_csv(csv_path, index=False)
            print(f"Saved paired metadata to {csv_path} ({len(df)} series pairs)")

        # train/val split using row_indices
        full_df = pd.read_csv(csv_path)
        n = len(full_df)
        indices = list(range(n))
        rng = random.Random(seed)
        rng.shuffle(indices)
        split = int(n * (1 - val_frac))
        selected = indices[:split] if section == "training" else indices[split:]

        super().__init__(src=csv_path, row_indices=selected, transform=transform)

    def _download_series(self, series_uid: str) -> str:
        """Download a DICOM series if not already present, return local directory path."""
        series_dir = os.path.join(self.img_dir, series_uid)
        if not os.path.exists(series_dir) or not os.listdir(series_dir):
            self.client.download_from_selection(
                seriesInstanceUID=series_uid,
                downloadDir=self.img_dir,
                dirTemplate="%SeriesInstanceUID",
            )
        return series_dir

    def __getitem__(self, index):
        if isinstance(index, int):
            row = self.data[index]
            row["image"] = self._download_series(row["image_SeriesInstanceUID"])
            row["seg"] = self._download_series(row["seg_SeriesInstanceUID"])
        return super().__getitem__(index=index)

## Create IDCDataset

We instantiate `IDCDataset` for the `qin_prostate_repeatability` collection. On first run, it queries IDC to build the paired metadata CSV and then downloads each DICOM series on demand as items are accessed.

In [ ]:
transform = Compose(
    [
        LoadImaged(reader="PydicomReader", keys=["image", "seg"]),
    ]
)

ds = IDCDataset(
    collection_id=collection_id,
    img_dir=os.path.join(root_dir, "idc_images"),
    section="training",
    val_frac=0.2,
    transform=transform,
)
print(f"Training set size: {len(ds)}")

### Detect dataset

Let's inspect the affine matrices, shapes, and available segment labels of the first sample. The image and segmentation are separate DICOM series, so their spatial metadata may differ.

In [ ]:
sample = 0

ds[sample]["image"].affine

In [ ]:
ds[sample]["seg"].affine

In [ ]:
print(ds[sample]["image"].shape, ds[sample]["seg"].shape)

### Inspect available segment labels

`PydicomReader` populates `seg.meta["labels"]` as a `dict` mapping segment name → channel index, derived directly from the DICOM `SegmentLabel` tags. Because this ordering is not guaranteed to be consistent across segmentations from different patients, **we must always look up segments by name** rather than by a hardcoded index.

In [ ]:
# Show all available segment labels for the first sample
labels = ds[sample]["seg"].meta["labels"]
print("Available segments:", labels)
print("\nSegment names:", list(labels.keys()))

As we can see from the metadata, the affine matrices between images and segmentations have some differences: the direction of the third spatial dimension is opposite.

We can use the `ResampleToMatchd` transform to unify them.
In addition, images do not have the channel dimension, and the channel dimension of segmentations can be adjusted into the first dimension. We can use `EnsureChannelFirstd` to address this issue.

### Add pre-processing transforms and reload

In [ ]:
transform = Compose(
    [
        LoadImaged(reader="PydicomReader", keys=["image", "seg"]),
        EnsureChannelFirstd(keys=["image", "seg"]),
        ResampleToMatchd(keys="image", key_dst="seg"),
    ]
)

ds = IDCDataset(
    collection_id=collection_id,
    img_dir=os.path.join(root_dir, "idc_images"),
    section="training",
    val_frac=0.2,
    transform=transform,
)

In [ ]:
sample = 0

ds[sample]["image"].affine

In [ ]:
ds[sample]["seg"].affine

In [ ]:
print(ds[sample]["image"].shape, ds[sample]["seg"].shape)

As we can see, now the affine matrices for images and segmentations are identical.

### Visualize and check

In [ ]:
whole_gland_idx = ds[4]["seg"].meta["labels"]["WholeGland"]
fig_seg = monai.visualize.matshow3d(seg[whole_gland_idx], show=True, every_n=every_n)

In [ ]:
every_n = 10
fig_img = monai.visualize.matshow3d(img[0], show=True, every_n=every_n)

In [ ]:
fig_seg = monai.visualize.matshow3d(seg[3], show=True, every_n=every_n)

## Create a Training Pipeline

In [ ]:
# Inspect available segment labels from the first training sample.
# seg.meta["labels"] maps segment name -> channel index; the ordering may differ
# across patients so we always resolve the target segment by name.
sample_labels = train_ds[0][0]["seg"].meta["labels"]
print("Available segments (name -> channel index):")
for name, idx in sample_labels.items():
    print(f"  {idx}: {name}")

# Select the target segment by name for training
target_segment = "WholeGland"
selected_label = sample_labels[target_segment]
print(f"\nTraining target: '{target_segment}' -> channel {selected_label}")

In [ ]:
train_transforms = Compose(
    [
        LoadImaged(reader="PydicomReader", keys=["image", "seg"], image_only=True),
        EnsureChannelFirstd(keys=["image", "seg"]),
        ResampleToMatchd(keys="image", key_dst="seg"),
        Orientationd(keys=["image", "seg"], axcodes="RAS"),
        Spacingd(keys=["image", "seg"], pixdim=(1.0, 1.0, 0.5), mode=("bilinear", "nearest")),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        RandCropByPosNegLabeld(
            keys=["image", "seg"],
            label_key="seg",
            spatial_size=(96, 96, 64),
            pos=1,
            neg=1,
            num_samples=2,
            image_key="image",
            image_threshold=0,
        ),
        RandFlipd(keys=("image", "seg"), prob=0.5, spatial_axis=[0]),
        RandFlipd(keys=("image", "seg"), prob=0.5, spatial_axis=[1]),
        RandScaleIntensityd(keys="image", factors=(-0.2, 0.2), prob=0.5),
        RandShiftIntensityd(keys="image", offsets=(-0.1, 0.1), prob=0.5),
    ]
)

val_transforms = Compose(
    [
        LoadImaged(reader="PydicomReader", keys=["image", "seg"], image_only=True),
        EnsureChannelFirstd(keys=["image", "seg"]),
        ResampleToMatchd(keys="image", key_dst="seg"),
        Orientationd(keys=["image", "seg"], axcodes="RAS"),
        Spacingd(keys=["image", "seg"], pixdim=(1.0, 1.0, 0.5), mode=("bilinear", "nearest")),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    ]
)

img_dir = os.path.join(root_dir, "idc_images")

train_ds = IDCDataset(
    collection_id=collection_id,
    img_dir=img_dir,
    section="training",
    val_frac=0.2,
    transform=train_transforms,
)

val_ds = IDCDataset(
    collection_id=collection_id,
    img_dir=img_dir,
    section="validation",
    val_frac=0.2,
    transform=val_transforms,
)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=1)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm="batch",
).to(device)
loss_function = DiceCELoss(sigmoid=True, include_background=True, batch=True, squared_pred=True)
optimizer = torch.optim.Adam(model.parameters(), 3e-4)
max_epochs = 1000
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
dice_metric = DiceMetric(include_background=True, reduction="mean")

### Create Model, Loss, Optimizer

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm="batch",
).to(device)
loss_function = DiceCELoss(sigmoid=True, include_background=True, batch=True, squared_pred=True)
optimizer = torch.optim.Adam(model.parameters(), 3e-4)
max_epochs = 1000
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
dice_metric = DiceMetric(include_background=True, reduction="mean")
selected_label = 3

### Execute a typical PyTorch training process

In [ ]:
val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = (
            batch_data["image"].to(device),
            batch_data["seg"][:, selected_label : selected_label + 1, ...].to(device),
        )
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    lr_scheduler.step()
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = (
                    val_data["image"].to(device),
                    val_data["seg"][:, selected_label : selected_label + 1, ...].to(device),
                )
                roi_size = (96, 96, 64)
                sw_batch_size = 4
                val_outputs = sliding_window_inference(val_inputs, roi_size, sw_batch_size, model)
                val_outputs = [post_pred(i) for i in decollate_batch(val_outputs)]
                dice_metric(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            dice_metric.reset()

            metric_values.append(metric)
            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), os.path.join(root_dir, "best_metric_model.pth"))
                print("saved new best metric model")
            print(
                f"current epoch: {epoch + 1} current mean dice: {metric:.4f}"
                f"\nbest mean dice: {best_metric:.4f} "
                f"at epoch: {best_metric_epoch}"
            )

In [ ]:
print(f"train completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")

## Plot the loss and metric

In [ ]:
plt.figure("train", (12, 6))
plt.subplot(1, 2, 1)
plt.title("Epoch Average Loss")
x = [i + 1 for i in range(len(epoch_loss_values))]
y = epoch_loss_values
plt.xlabel("epoch")
plt.plot(x, y)
plt.subplot(1, 2, 2)
plt.title("Val Mean Dice")
x = [val_interval * (i + 1) for i in range(len(metric_values))]
y = metric_values
plt.xlabel("epoch")
plt.plot(x, y)
plt.show()

## Check best model output with the input image and label

In [ ]:
model.load_state_dict(torch.load(os.path.join(root_dir, "best_metric_model.pth"), weights_only=True))
model.eval()
with torch.no_grad():
    for i, val_data in enumerate(val_loader):
        roi_size = (96, 96, 64)
        sw_batch_size = 4
        val_outputs = sliding_window_inference(val_data["image"].to(device), roi_size, sw_batch_size, model)
        depth = val_data["image"].shape[-1]
        slc = depth // 2
        plt.figure("check", (18, 6))
        plt.subplot(1, 3, 1)
        plt.title(f"image {i}")
        plt.imshow(val_data["image"][0, 0, :, :, slc], cmap="gray")
        plt.subplot(1, 3, 2)
        plt.title(f"label {i}")
        plt.imshow(val_data["seg"][0, selected_label, :, :, slc])
        plt.subplot(1, 3, 3)
        plt.title(f"output {i}")
        plt.imshow(post_pred(val_outputs[0]).detach().cpu()[0, :, :, slc])
        plt.show()
        if i == 2:
            break

## Cleanup data directory

Remove directory if a temporary was used.

In [ ]:
if directory is None:
    shutil.rmtree(root_dir)

# Acknowledgments

This notebook uses the **QIN-PROSTATE-Repeatability** dataset hosted on NCI Imaging Data Commons. Users must abide by the [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/) under which it has been published. Attribution should include references to the following citations:

* Fedorov, A; Schwier, M; Clunie, D; Herz, C; Pieper, S; Kikinis, R; Tempany, C; Fennessy, F. (2018). Data From QIN-PROSTATE-Repeatability. The Cancer Imaging Archive. DOI: [10.7937/K9/TCIA.2018.MR1CKGND](http://doi.org/10.7937/K9/TCIA.2018.MR1CKGND)

* Fedorov A, Vangel MG, Tempany CM, Fennessy FM. Multiparametric Magnetic Resonance Imaging of the Prostate: Repeatability of Volume and Apparent Diffusion Coefficient Quantification. Investigative Radiology. 52, 538–546 (2017). DOI: [10.1097/RLI.0000000000000382](https://doi.org/10.1097/RLI.0000000000000382)

* Fedorov, A., Schwier, M., Clunie, D., Herz, C., Pieper, S., Kikinis, R., Tempany, C. & Fennessy, F. An annotated test-retest collection of prostate multiparametric MRI. Scientific Data 5, 180281 (2018). DOI: [10.1038/sdata.2018.281](https://doi.org/10.1038/sdata.2018.281)

* Clark K, Vendt B, Smith K, Freymann J, Kirby J, Koppel P, Moore S, Phillips S, Maffitt D, Pringle M, Tarbox L, Prior F. The Cancer Imaging Archive (TCIA): Maintaining and Operating a Public Information Repository, Journal of Digital Imaging, Volume 26, Number 6, December, 2013, pp 1045-1057. DOI: [10.1007/s10278-013-9622-7](https://dx.doi.org/10.1007%2Fs10278-013-9622-7)

When using IDC data in publications, please also cite:

* Fedorov, A., Longabaugh, W. J. R., Pot, D., Clunie, D. A., Pieper, S. D., Gibbs, D. L., Bridge, C., Herrmann, M. D., Homeyer, A., Lewis, R., Agrawal, P., Krishnaswamy, D., Thiriveedhi, V. K., Ciausu, C., Schacherer, D. P., Bontempi, D., Pihl, T., Wagner, U., Farahani, K., Kim, E., & Kikinis, R. National Cancer Institute Imaging Data Commons: Toward Transparency, Reproducibility, and Scalability in Imaging Artificial Intelligence. RadioGraphics 43(12), (2023). DOI: [10.1148/rg.230180](https://doi.org/10.1148/rg.230180)